<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [4]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    print("Carregando os dados")
    X, y = fetch_openml('mnist_784', version=1, as_frame=False, return_X_y=True)

    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    
    return X_train, X_test, y_train, y_test

Não é estritamente necessário normalizar os dados para modelos baseados em árvores de decisão, como Random Forest e Gradient Boosted Trees (AdaBoost). Esses algoritmos tomam decisões baseadas em regras de corte (splits) em características individuais, independentemente da escala dos dados. A normalização não afeta a estrutura da árvore construída nem a sua capacidade preditiva.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [5]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    rf_model = RandomForestClassifier(random_state=seed)

    rf_model.fit(X_train, y_train)
    
    return rf_model

def train_adaboost(X_train, y_train, seed=42):
    ab_model = AdaBoostClassifier(random_state=seed)

    ab_model.fit(X_train, y_train)
    
    return ab_model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [6]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    
    return accuracy

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [9]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)

    accuracy = evaluate(model, X_test, y_test)

    return accuracy

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

Em qual profundidade começa o overfitting?
O overfitting começa quando a árvore se torna profunda o suficiente para memorizar ruídos e detalhes específicos do conjunto de treino em vez de aprender o padrão geral. Na prática, isso é observado quando a acurácia de treino continua subindo, mas a acurácia de teste estabiliza ou começa a cair. Em datasets complexos como imagens (MNIST), isso geralmente começa a ficar evidente em profundidades maiores (frequentemente a partir de 10 a 15, dependendo dos dados).

Por que a árvore consegue 100% no treino quando max_depth=None?
Quando definimos max_depth=None, não impomos nenhum limite de profundidade para a árvore. Isso permite que ela continue dividindo os dados (criando novos "galhos") até que todas as folhas sejam "puras" — ou seja, até que cada folha contenha apenas amostras de uma única classe (ou até chegar no limite mínimo de amostras por divisão). Na prática, o modelo cria uma regra específica para cada exemplo do conjunto de treinamento, decorando os dados e atingindo 100% de acurácia nele.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Carregando dados...")
X_train, X_test, y_train, y_test = load_data(seed=42)

print("\nTreinando Random Forest...")
rf_model = train_random_forest(X_train, y_train, seed=42)
y_pred_rf = rf_model.predict(X_test)

acc_rf = evaluate(rf_model, X_test, y_test)
prec_rf = precision_score(y_test, y_pred_rf, average='weighted')
rec_rf = recall_score(y_test, y_pred_rf, average='weighted')
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

print("--- Resultados Random Forest ---")
print(f"Acurácia: {acc_rf:.4f}")
print(f"Precisão: {prec_rf:.4f}")
print(f"Recall:   {rec_rf:.4f}")
print(f"F1-Score: {f1_rf:.4f}")

print("\nTreinando AdaBoost...")
ab_model = train_adaboost(X_train, y_train, seed=42)
y_pred_ab = ab_model.predict(X_test)

acc_ab = evaluate(ab_model, X_test, y_test)
prec_ab = precision_score(y_test, y_pred_ab, average='weighted')
rec_ab = recall_score(y_test, y_pred_ab, average='weighted')
f1_ab = f1_score(y_test, y_pred_ab, average='weighted')

print("\n--- Resultados AdaBoost ---")
print(f"Acurácia: {acc_ab:.4f}")
print(f"Precisão: {prec_ab:.4f}")
print(f"Recall:   {rec_ab:.4f}")
print(f"F1-Score: {f1_ab:.4f}")

Carregando dados...
Carregando os dados

Treinando Random Forest...
--- Resultados Random Forest ---
Acurácia: 0.9673
Precisão: 0.9673
Recall:   0.9673
F1-Score: 0.9673

Treinando AdaBoost...

--- Resultados AdaBoost ---
Acurácia: 0.6426
Precisão: 0.6852
Recall:   0.6426
F1-Score: 0.6483


O Random Forest apresentou um desempenho inicial significativamente superior. Isso ocorre porque o Random Forest, por padrão, utiliza árvores de decisão profundas e complexas como estimadores base, conseguindo capturar rapidamente as interações complexas dos pixels nas imagens. Já o AdaBoost do sklearn utiliza Decision Stumps (árvores de profundidade 1) por padrão. Como imagens são dados complexos, "stumps" são muito fracos para resolver o problema rapidamente sem um ajuste rigoroso de hiperparâmetros (como aumentar o número de estimadores ou a profundidade do estimador base).


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

Os resultados mudaram?
Sim, ao alterar a seed (de 42 para 7), a acurácia sofreu uma leve variação. Isso acontece porque a semente aleatória controla dois fatores cruciais: a forma como os dados são embaralhados e divididos entre treino e teste (no train_test_split), e a aleatoriedade interna do próprio algoritmo (como o bootstrapping de amostras e a seleção de features no Random Forest).

O experimento é reprodutível? Justifique.
Sim, o experimento é perfeitamente reprodutível. A reprodutibilidade em machine learning não significa que o modelo dará o mesmo resultado para qualquer semente, mas sim que, ao fixarmos uma semente específica (ex: 42), garantimos o controle total sobre todos os processos estocásticos (aleatórios) do pipeline. Qualquer pessoa que rodar este notebook com a seed 42 obterá exatamente os mesmos números, o que torna o experimento confiável e auditável.

In [10]:
print("Executando pipeline com Random Forest...")

acc_seed_42 = run_pipeline(model_type="rf", seed=42)
print(f"Acurácia com seed 42: {acc_seed_42:.4f}")

acc_seed_7 = run_pipeline(model_type="rf", seed=7)
print(f"Acurácia com seed 7:  {acc_seed_7:.4f}")

acc_seed_42_repetida = run_pipeline(model_type="rf", seed=42)
print(f"Acurácia repetindo seed 42: {acc_seed_42_repetida:.4f}")

Executando pipeline com Random Forest...
Carregando os dados
Acurácia com seed 42: 0.9673
Carregando os dados
Acurácia com seed 7:  0.9717
Carregando os dados
Acurácia repetindo seed 42: 0.9673


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [11]:
print("Carregando dados...")
X_train, X_test, y_train, y_test = load_data(seed=42)

print("\n--- Random Forest ---")
rf_model = train_random_forest(X_train, y_train, seed=42)

acc_train_rf = evaluate(rf_model, X_train, y_train)
acc_test_rf = evaluate(rf_model, X_test, y_test)

print(f"Acurácia no Treino: {acc_train_rf:.4f}")
print(f"Acurácia no Teste:  {acc_test_rf:.4f}")

print("\n--- AdaBoost ---")
ab_model = train_adaboost(X_train, y_train, seed=42)

acc_train_ab = evaluate(ab_model, X_train, y_train)
acc_test_ab = evaluate(ab_model, X_test, y_test)

print(f"Acurácia no Treino: {acc_train_ab:.4f}")
print(f"Acurácia no Teste:  {acc_test_ab:.4f}")

Carregando dados...
Carregando os dados

--- Random Forest ---
Acurácia no Treino: 1.0000
Acurácia no Teste:  0.9673

--- AdaBoost ---
Acurácia no Treino: 0.6426
Acurácia no Teste:  0.6426


Existe overfitting?
Sim, existe um overfitting claro, principalmente no Random Forest. A acurácia no conjunto de treino chega a 100% (1.0), enquanto no conjunto de teste cai para cerca de 96.7%. Isso mostra que o modelo praticamente "memorizou" os detalhes exatos e os ruídos do conjunto de treinamento, não conseguindo generalizar perfeitamente para os dados de teste.

Qual modelo tende a sofrer mais com isso?
O Random Forest. Por padrão, no scikit-learn, o hiperparâmetro max_depth vem configurado como None. Isso significa que as árvores de decisão dentro da "floresta" crescem sem limites até que todas as folhas contenham apenas amostras de uma única classe (puras). Isso faz com que o modelo decore os dados de treino com extrema facilidade. Já o AdaBoost, por utilizar "Decision Stumps" (árvores de profundidade 1) como padrão, é um modelo muito mais simples e tende a sofrer mais com underfitting (não conseguir aprender a complexidade dos dados, ficando com ~64% em ambos) do que com overfitting neste cenário inicial.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [12]:
print("Carregando dados...")
X_train, X_test, y_train, y_test = load_data(seed=42)

valores_n_estimators = [10, 50, 100]

print("\n--- Testando Random Forest ---")
for n in valores_n_estimators:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    acc = evaluate(rf, X_test, y_test)
    print(f"n_estimators={n:3d} | Acurácia: {acc:.4f}")

print("\n--- Testando AdaBoost ---")
for n in valores_n_estimators:
    ab = AdaBoostClassifier(n_estimators=n, random_state=42)
    ab.fit(X_train, y_train)
    acc = evaluate(ab, X_test, y_test)
    print(f"n_estimators={n:3d} | Acurácia: {acc:.4f}")

Carregando dados...
Carregando os dados

--- Testando Random Forest ---
n_estimators= 10 | Acurácia: 0.9456
n_estimators= 50 | Acurácia: 0.9639
n_estimators=100 | Acurácia: 0.9673

--- Testando AdaBoost ---
n_estimators= 10 | Acurácia: 0.2428
n_estimators= 50 | Acurácia: 0.6426
n_estimators=100 | Acurácia: 0.7174


O desempenho muda significativamente?
Sim, o desempenho muda, mas de formas diferentes para cada modelo. No Random Forest, o salto maior acontece ao sair de um número muito baixo (como 10) para um número intermediário (50). A partir de 100 árvores, o ganho de acurácia se torna marginal (estabiliza). No AdaBoost, o aumento de n_estimators tende a mostrar melhorias mais contínuas e proporcionais no início, pois ele precisa de muitos ciclos para que os estimadores fracos (stumps) consigam juntos resolver a complexidade do dataset.

Qual modelo é mais sensível a mudanças?
O AdaBoost. Como ele funciona de forma sequencial (onde cada novo estimador foca nos erros do anterior), o número de estimadores dita diretamente a capacidade do modelo de aprender limites de decisão mais complexos. O Random Forest, por usar árvores independentes e profundas desde o início, é menos sensível e atinge um bom desempenho muito mais rápido, usando mais árvores apenas para reduzir a variância (estabilidade) do que para aprender novos padrões complexos.

In [ ]:
# TODO: implemente load_data

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. A acurácia é suficiente para avaliar os modelos?
Não. Embora o Fashion MNIST seja um dataset relativamente balanceado, a acurácia isolada pode mascarar problemas em classes específicas. Por exemplo, o modelo pode ser excelente em classificar "camisetas", mas péssimo em diferenciar "tênis" de "botas". É por isso que calculamos a Precisão, o Recall e o F1-Score na Questão 5, pois essas métricas nos ajudam a entender o comportamento do modelo em relação aos falsos positivos e falsos negativos de cada classe individualmente.

2. Como você garante que o resultado não ocorreu por acaso?
Garantimos isso através do controle rigoroso da aleatoriedade do experimento. Ao fixarmos o parâmetro random_state (nossa seed igual a 42) tanto na divisão dos dados (no train_test_split) quanto na inicialização dos modelos (Random Forest e AdaBoost), forçamos o algoritmo a tomar sempre as mesmas decisões probabilísticas. Isso torna o experimento determinístico e 100% reprodutível, provando que o desempenho obtido é mérito da capacidade de aprendizado do modelo e não de uma "sorte" na separação dos dados.

3. Cite dois possíveis problemas metodológicos neste experimento.
O primeiro problema é a ausência de um conjunto de Validação. Na Questão 8, testamos hiperparâmetros (como n_estimators) e avaliamos o resultado direto no conjunto de Teste. Isso é considerado Data Leakage (vazamento de dados), pois estamos tomando decisões de modelagem baseadas nos dados que deveriam ser inéditos. O correto seria dividir os dados em Treino, Validação (para testar parâmetros) e Teste (apenas para a nota final).
O segundo problema é a falta de Validação Cruzada (Cross-Validation). Como dividimos os dados apenas uma vez, nosso resultado está muito atrelado àquela divisão específica. A validação cruzada nos daria uma média de desempenho muito mais robusta.

4. O pipeline implementado é confiável? Justifique.
Ele é parcialmente confiável. Do ponto de vista da engenharia de software, ele é excelente: é modular, limpo, tem funções bem definidas e é totalmente reprodutível graças ao controle de sementes (passando liso no CI do GitHub Actions). No entanto, do ponto de vista de Machine Learning estrito, ele peca por não utilizar um conjunto de validação para o ajuste fino dos hiperparâmetros, o que pode levar a uma estimativa excessivamente otimista do desempenho real do modelo em produção.